# Project: Job Market Optimization Analysis
**Persona:** Persona D — The Job Seeker Optimisation Coach

## Project Overview
This project analyzes job posting metadata to provide data-driven insights for job seekers. We focus on timing, title optimization, and identifying low-competition niches in the market.

## Rerferences
1. https://docs.google.com/document/d/1Jm8-oaM8JDiAe1Z044rTMMhMGb8Cg6bIsLIZKBnCrwM/edit?tab=t.0


## Persona D — The Job Seeker Optimisation Coach
"I help job seekers compete. I want to know: when to apply, what title patterns get the most views, and where applications are most likely to convert."

- Use dt.day_name() and dt.month on metadata_originalPostingDate to find which posting days and months attract the most views and applications.
- Use str.contains to flag titles with keywords like Senior, Junior, Lead, Specialist, Manager. Compare median views and application counts across these groups.
- Which primary_category has the lowest applications-per-vacancy ratio (least competitive)? Visualise as a horizontal bar chart.

## Business Questions
1. **When to Apply?** Which days of the week and months of the year do job postings receive the most views and applications?
2. **Title Optimization?** Do certain keywords in job titles (e.g. Senior, Junior, Lead) correlate with higher views and application counts?        
3. **Low-Competition Niches?** Which primary job categories have the lowest applications-per-vacancy ratio, indicating less competition?

## Additional Questions to Explore
4) **"Hidden Gems"—categories with a high applications-per-vacancy ratio but low total applications, suggesting they are overlooked by job seekers.**
5) **"Sweet Spot"—categories where the salary is high but the competition (apps-per-vacancy) is relatively low.**
6) What skill set is required for the job?

## Stretch challenges (if you finish early)
 - Salary band width analysis: For each primary_category, compute salary_maximum - salary_minimum using NumPy. Which categories have the widest bands? What might that signal about negotiation room?
- "Ghost jobs" revisited: Using Pandas date arithmetic (metadata_expiryDate - metadata_originalPostingDate), identify postings open for more than 90 days with zero applications. How prevalent are they? Which categories and levels?
- Title keyword co-occurrence: Build a frequency table of two-word title bigrams (split title on spaces, then use pd.Series(bigrams).value_counts()). Which bigrams are trending upward over time?
- Cohort analysis: Group jobs by posting quarter. For each cohort, track median salary, median minimumYearsExperience, and application rate. Has the market become more or less demanding over time?
- Agency concentration index: For each primary_category, compute the share of jobs posted by the top 3 agencies. Build a "concentration index" and rank categories from most agency-dominated to least.

## Data Dictionary
<details>
<summary>Click to view full column metadata</summary>

| Column | Type | Notes |
| :--- | :--- | :--- |
| **categories** | `str (JSON)` | JSON array of categories. Needs parsing. |
| **employmentTypes** | `str` | Type of contract (Permanent, Full Time, etc.) |
| **metadata_originalPostingDate** | `date` | Original date the job was listed. |
| **metadata_repostCount** | `int` | Signal of hard-to-fill roles. |
| **metadata_totalNumberOfView** | `int` | Total views received. |
| **metadata_totalNumberJobApplication** | `int` | Total applications received. |
| **positionLevels** | `str` | Seniority (Entry, Executive, Manager, etc.) |
| **salary_minimum / maximum** | `int` | Monthly salary range. |
| **average_salary** | `float` | Mean of min/max salary. |

*(Table truncated for brevity—paste the full version here)*
</details>

## Technical requirements (your notebook must demonstrate ALL of these)
To prove you stretched your Python muscles, your final notebook must include:

- [ ] At least one NumPy operation applied to a column extracted as an array (e.g. np.mean, np.percentile, np.std, boolean mask, or a normalisation formula).
- [ ] At least one Pandas groupby with a meaningful aggregation (not just .count()) — e.g. .agg({'col': ['mean', 'median']}).
- [ ] At least one missing-value handling step — isnull() inspection plus a .fillna(), .dropna(), or .replace() decision that is documented in a Markdown cell.
- [ ] At least one string/text operation on the title or positionLevels or categories column (.str.contains, .str.extract, .str.upper, etc.).
- [ ] At least one date/time operation using the metadata_originalPostingDate column (dt.month, dt.year, dt.to_period, etc.).
- [ ] At least two different chart types (e.g. bar + boxplot, histogram + scatter, line + heatmap) — each with a title, axis labels, and a one-sentence interpretation comment below.
- [ ] A documented data-cleaning step in a Markdown cell: what you removed, why, and how many rows were affected.

### source data


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


df = pd.read_csv('../data/SGJobData.csv')
df.info()





<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048585 entries, 0 to 1048584
Data columns (total 22 columns):
 #   Column                              Non-Null Count    Dtype  
---  ------                              --------------    -----  
 0   categories                          1044597 non-null  object 
 1   employmentTypes                     1044597 non-null  object 
 2   metadata_expiryDate                 1044597 non-null  object 
 3   metadata_isPostedOnBehalf           1048585 non-null  bool   
 4   metadata_jobPostId                  1044597 non-null  object 
 5   metadata_newPostingDate             1044597 non-null  object 
 6   metadata_originalPostingDate        1044597 non-null  object 
 7   metadata_repostCount                1048585 non-null  int64  
 8   metadata_totalNumberJobApplication  1048585 non-null  int64  
 9   metadata_totalNumberOfView          1048585 non-null  int64  
 10  minimumYearsExperience              1048585 non-null  int64  
 11  numberOfVac

### Initial Profiling

In [ ]:
# initial profiling

# Shape, types, missing values at a glance
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]}")
print("\nNull counts:\n", df.isnull().sum().sort_values(ascending=False).head(10))
print("\nBasic stats:\n", df[['salary_minimum','salary_maximum','average_salary',
                               'minimumYearsExperience','numberOfVacancies']].describe())

Rows: 1,048,585  Columns: 22

Null counts:
 occupationId                    1048585
categories                         3988
metadata_expiryDate                3988
title                              3988
metadata_jobPostId                 3988
metadata_newPostingDate            3988
metadata_originalPostingDate       3988
status_jobStatus                   3988
salary_type                        3988
employmentTypes                    3988
dtype: int64

Basic stats:
        salary_minimum  salary_maximum  average_salary  minimumYearsExperience  \
count    1.048585e+06    1.048585e+06    1.048585e+06            1.048585e+06   
mean     3.815312e+03    5.723578e+03    4.769445e+03            2.779573e+00   
std      3.172182e+03    5.018387e+04    2.547809e+04            2.537049e+00   
min      0.000000e+00    0.000000e+00    0.000000e+00            0.000000e+00   
25%      2.500000e+03    3.300000e+03    2.900000e+03            1.000000e+00   
50%      3.000000e+03    4.500000e+03    3

### Salary cleaning example


In [3]:
# Remove obvious outliers using the 99th percentile
p99 = np.percentile(df['average_salary'].dropna(), 99)
df_clean = df[(df['average_salary'] > 500) & (df['average_salary'] <= p99)].copy()

# Use NumPy to compute stats on the cleaned array
sal = df_clean['average_salary'].to_numpy()
print(f"Mean: {np.mean(sal):,.0f}  Median: {np.median(sal):,.0f}  Std: {np.std(sal):,.0f}")
print(f"25th pct: {np.percentile(sal, 25):,.0f}  75th pct: {np.percentile(sal, 75):,.0f}")


Mean: 4,551  Median: 3,800  Std: 2,533
25th pct: 2,900  75th pct: 5,500
